# 18xC — Controlled unblinding and trading simulation

This stage verifies the frozen 18xB signal release before loading realised HKO outcomes. A trade buys one YES share at the observed pre-cutoff price proxy. Net profit is \(Y-p^{market}-c\), where \(c\) is a hypothetical all-in cost per share. The simulation reports turnover, profit and loss, return on committed capital, hit rate and drawdown.

**Revision v2.** Maximum drawdown now includes the initial portfolio value of zero as the first running peak. PnL, return, hit rate, trade count and all signals remain unchanged.

In [1]:
from __future__ import annotations
import hashlib, json, math, platform, sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
ROOT=Path.cwd().resolve()
if not (ROOT/'.git').exists(): raise RuntimeError(f'Run from repository root, not {ROOT}')
UTC=timezone.utc

def sha(path:Path)->str:
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def parse_bool(s:pd.Series,name:str)->pd.Series:
    if pd.api.types.is_bool_dtype(s): return s.astype(bool)
    out=s.astype(str).str.strip().str.lower().map({'true':True,'false':False,'1':True,'0':False,'yes':True,'no':False})
    if out.isna().any(): raise ValueError(f'Cannot parse Boolean {name}: {s[out.isna()].drop_duplicates().tolist()}')
    return out.astype(bool)

def verify_manifest(path:Path):
    m=pd.read_csv(path); failures=[]
    for r in m.itertuples(index=False):
        p=ROOT/r.path
        if not p.is_file(): failures.append(f'MISSING {r.path}'); continue
        if sha(p)!=r.sha256: failures.append(f'HASH {r.path}')
        if p.stat().st_size!=int(r.size_bytes): failures.append(f'SIZE {r.path}')
    if failures: raise AssertionError(f'Manifest failed {path}:\\n'+'\\n'.join(failures))

def write_manifest(out:Path, report_dir:Path, filename:str):
    rows=[]
    for root in [out,report_dir]:
        for p in sorted(root.rglob('*')):
            if p.is_file() and p.name!=filename:
                rows.append({'path':str(p.relative_to(ROOT)),'size_bytes':p.stat().st_size,'sha256':sha(p)})
    pd.DataFrame(rows).to_csv(out/filename,index=False)

def save_frame(frame:pd.DataFrame,path:Path):
    x=frame.copy()
    for c in x.columns:
        if pd.api.types.is_datetime64_any_dtype(x[c]):
            if getattr(x[c].dt,'tz',None) is not None: x[c]=x[c].astype('string')
            else: x[c]=x[c].dt.strftime('%Y-%m-%d')
    x.to_csv(path,index=False)

def max_drawdown(daily:pd.Series)->float:
    if daily.empty: return float('nan')
    cumulative=daily.sort_index().cumsum().to_numpy(dtype=float)
    running_peak=np.maximum.accumulate(np.concatenate(([0.0],cumulative)))[1:]
    return float(np.min(cumulative-running_peak))

STEP='18xC'; COSTS=np.array([0.00,0.005,0.01,0.02,0.05]); PRIMARY_COST=0.01
XB=ROOT/'data/processed/18xB_blind_trading_signal_release'; S=ROOT/'data/processed/18s_expanded_march_june_canonical_sample'
SIGNALS=XB/'18xB_blind_trading_signal_panel.csv'; XB_SUM=XB/'18xB_summary.json'; XB_MAN=XB/'18xB_sha256_manifest.csv'; XB_PROTOCOL=XB/'18xB_protocol.json'; XB_SOURCES=XB/'18xB_source_inventory.csv'
OUTCOMES=S/'18s_expanded_certified_contract_outcome_panel.csv'; S_SUM=S/'18s_expanded_sample_summary.json'; S_MAN=S/'18s_expanded_sha256_manifest.csv'
OUT=ROOT/'data/processed/18xC_controlled_unblinding_trading_simulation'; REPORT=ROOT/'reports/18xC_controlled_unblinding_trading_simulation'; OUT.mkdir(parents=True,exist_ok=True); REPORT.mkdir(parents=True,exist_ok=True)
for p in [SIGNALS,XB_SUM,XB_MAN,XB_PROTOCOL,XB_SOURCES,OUTCOMES,S_SUM,S_MAN]:
    if not p.is_file(): raise FileNotFoundError(p)
verify_manifest(XB_MAN); verify_manifest(S_MAN)
if json.loads(XB_SUM.read_text()).get('verdict')!='PASS' or json.loads(S_SUM.read_text()).get('verdict')!='PASS': raise AssertionError('Upstream not PASS')
xb_protocol=json.loads(XB_PROTOCOL.read_text());
if xb_protocol.get('outcomes_loaded') is not False: raise AssertionError('18xB is not outcome blind')
forbidden_sources=['certified_contract_outcome','market_scoring','model_probability_outcome']
source_text=' '.join(pd.read_csv(XB_SOURCES).path.astype(str)).lower()
if any(x in source_text for x in forbidden_sources): raise AssertionError('18xB source inventory contains outcome path')
signals=pd.read_csv(SIGNALS,dtype={'market_id':str},low_memory=False); outcomes=pd.read_csv(OUTCOMES,dtype={'market_id':str},low_memory=False)
for d in [signals,outcomes]: d['event_date']=pd.to_datetime(d.event_date,errors='raise')
if len(signals)!=115 or not parse_bool(signals.outcome_blind_signal,'outcome_blind_signal').all(): raise AssertionError('Invalid frozen signals')
outcome=outcomes[['event_date','market_id','hko_daily_max_c','Y_event_int','Y_no_int']].drop_duplicates(['event_date','market_id'])
panel=signals.merge(outcome,on=['event_date','market_id'],how='left',validate='many_to_one')
if panel.Y_event_int.isna().any(): raise AssertionError('Missing outcome after controlled unblinding')
panel['gross_pnl_per_share']=np.where(parse_bool(panel.trade_flag,'trade_flag'),panel.Y_event_int-panel.p_market,0.0); panel['winning_trade']=parse_bool(panel.trade_flag,'trade_flag')&panel.Y_event_int.eq(1)
trade_rows=[]; daily_rows=[]; summary_rows=[]
for (role,cid,variant,rule,block),g in panel.groupby(['strategy_role','candidate_id','probability_variant','decision_rule','evaluation_block'],sort=True):
    trade_flag=parse_bool(g.trade_flag,'trade_flag')
    for cost in COSTS:
        x=g.copy(); x['cost_per_share']=float(cost); x['net_pnl']=np.where(trade_flag,x.Y_event_int-x.p_market-cost,0.0); x['capital_committed']=np.where(trade_flag,x.p_market+cost,0.0); x['trade_hit']=trade_flag&x.Y_event_int.eq(1); x['primary_cost_flag']=np.isclose(cost,PRIMARY_COST); trade_rows.append(x)
        daily=x.groupby('event_date',as_index=False).agg(daily_net_pnl=('net_pnl','sum'),daily_capital=('capital_committed','sum'),trade_count=('trade_flag','sum')); daily['strategy_role']=role; daily['candidate_id']=cid; daily['probability_variant']=variant; daily['decision_rule']=rule; daily['evaluation_block']=block; daily['cost_per_share']=float(cost); daily=daily.sort_values('event_date'); daily['cumulative_net_pnl']=daily.daily_net_pnl.cumsum(); cumulative=daily.cumulative_net_pnl.to_numpy(dtype=float); daily['running_peak']=np.maximum.accumulate(np.concatenate(([0.0],cumulative)))[1:]; daily['drawdown']=daily.cumulative_net_pnl-daily.running_peak; daily['drawdown_reference_initial_value']=0.0; daily_rows.append(daily)
        trades=x.loc[trade_flag]; pnl=trades.net_pnl; capital=trades.capital_committed
        summary_rows.append({'strategy_role':role,'candidate_id':cid,'probability_variant':variant,'decision_rule':rule,'evaluation_block':block,'cost_per_share':float(cost),'opportunity_books':len(x),'opportunity_dates':x.event_date.nunique(),'trade_count':len(trades),'trade_dates':trades.event_date.nunique(),'trade_rate':len(trades)/len(x),'total_net_pnl':float(pnl.sum()),'mean_net_pnl_per_trade':float(pnl.mean()) if len(trades) else np.nan,'median_net_pnl_per_trade':float(pnl.median()) if len(trades) else np.nan,'total_capital':float(capital.sum()),'return_on_capital':float(pnl.sum()/capital.sum()) if capital.sum()>0 else np.nan,'hit_rate':float((pnl>0).mean()) if len(trades) else np.nan,'mean_edge_trades':float(trades.edge.mean()) if len(trades) else np.nan,'max_drawdown':float(daily.drawdown.min()),'largest_gain':float(pnl.max()) if len(trades) else np.nan,'largest_loss':float(pnl.min()) if len(trades) else np.nan,'break_even_cost_per_trade':float(trades.gross_pnl_per_share.sum()/len(trades)) if len(trades) else np.nan})
trade_panel=pd.concat(trade_rows,ignore_index=True); daily_panel=pd.concat(daily_rows,ignore_index=True); summary=pd.DataFrame(summary_rows)
if len(trade_panel)!=575 or len(summary)!=30: raise AssertionError(f'Unexpected simulation counts {len(trade_panel)} {len(summary)}')
primary=summary.loc[summary.cost_per_share.eq(PRIMARY_COST)].copy()
expected={('PRIMARY_OVERALL','INTERNAL_HOLDOUT'):0.2625,('PRIMARY_OVERALL','EXTERNAL_TEST'):-1.4925,('GAUSSIAN_PROCESS_FAMILY','INTERNAL_HOLDOUT'):0.5165,('GAUSSIAN_PROCESS_FAMILY','EXTERNAL_TEST'):-1.063,('TREE_FAMILY','INTERNAL_HOLDOUT'):-0.5505,('TREE_FAMILY','EXTERNAL_TEST'):-1.9275}
expected_drawdown={('PRIMARY_OVERALL','INTERNAL_HOLDOUT'):-0.5725,('PRIMARY_OVERALL','EXTERNAL_TEST'):-1.4925,('GAUSSIAN_PROCESS_FAMILY','INTERNAL_HOLDOUT'):-0.3730,('GAUSSIAN_PROCESS_FAMILY','EXTERNAL_TEST'):-1.1975,('TREE_FAMILY','INTERNAL_HOLDOUT'):-0.5505,('TREE_FAMILY','EXTERNAL_TEST'):-1.9275}
for k,v in expected.items():
    r=primary.loc[primary.strategy_role.eq(k[0])&primary.evaluation_block.eq(k[1])]
    if len(r)!=1 or not np.isclose(r.total_net_pnl.iloc[0],v): raise AssertionError(f'Unexpected PnL {k}: {r.to_dict("records")}')
    if not np.isclose(r.max_drawdown.iloc[0],expected_drawdown[k]): raise AssertionError(f'Unexpected corrected drawdown {k}: {r.max_drawdown.iloc[0]}')
checks=pd.DataFrame([{'check':'18xB_manifest_verified','passed':True,'detail':sha(XB_MAN),'blocking':True},{'check':'frozen_signal_rows_115','passed':len(panel)==115,'detail':str(len(panel)),'blocking':True},{'check':'trade_panel_rows_575','passed':len(trade_panel)==575,'detail':str(len(trade_panel)),'blocking':True},{'check':'cost_summary_rows_30','passed':len(summary)==30,'detail':str(len(summary)),'blocking':True},{'check':'primary_cost_rows_6','passed':len(primary)==6,'detail':str(len(primary)),'blocking':True},{'check':'raw_entry_prices_retained','passed':np.allclose(panel.assumed_entry_price,panel.p_market),'detail':'un-normalised observed price','blocking':True},{'check':'unblind_after_freeze','passed':True,'detail':'18xB protocol and source inventory checked before outcome load','blocking':True},{'check':'maximum_drawdown_includes_zero_initial_value','passed':all(np.isclose(primary.loc[primary.strategy_role.eq(k[0])&primary.evaluation_block.eq(k[1]),'max_drawdown'].iloc[0],v) for k,v in expected_drawdown.items()),'detail':'six corrected primary-cost drawdowns','blocking':True}])
issues=pd.DataFrame(columns=['issue_level','issue_code','strategy_role','event_date','decision_rule','detail','blocking'])
for name,frame in {'unblinded_signal_outcome_panel':panel,'cost_sensitivity_trade_panel':trade_panel,'daily_pnl_panel':daily_panel,'strategy_cost_summary':summary,'primary_cost_summary':primary,'integrity_checks':checks,'issues':issues}.items(): save_frame(frame,OUT/f'18xC_{name}.csv')
protocol={'step':STEP,'generated_at_utc':datetime.now(UTC).isoformat(),'verdict':'PASS','controlled_unblinding':'18xB manifest, protocol, source inventory and blind flag verified before loading outcome panel','pnl_per_one_yes_share':'Y - p_market - cost_per_share','cost_grid':COSTS.tolist(),'primary_cost_per_share':PRIMARY_COST,'execution_limitations':['selected observed pre-cutoff YES price is a fill proxy','no bid-ask quote','no liquidity or partial-fill model','no market impact','no position netting'],'market_prices_normalised':False,'maximum_drawdown_definition':'minimum cumulative PnL relative to running peak including initial portfolio value zero'}
(OUT/'18xC_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
summary_json={'step':STEP,'generated_at_utc':datetime.now(UTC).isoformat(),'verdict':'PASS','frozen_signal_rows':115,'trade_signals':97,'cost_sensitivity_trade_rows':575,'strategy_cost_summary_rows':30,'primary_cost_summary_rows':6,'primary_cost_results':primary[['strategy_role','candidate_id','evaluation_block','trade_count','total_net_pnl','return_on_capital','hit_rate','max_drawdown']].to_dict('records'),'issue_rows':0,'integrity_checks_passed':int(checks.passed.sum()),'integrity_checks_total':len(checks)}
(OUT/'18xC_summary.json').write_text(json.dumps(summary_json,indent=2),encoding='utf-8')
pd.DataFrame([{'input_role':'18xB_blind_trading_signal_panel','path':str(SIGNALS.relative_to(ROOT)),'rows':len(signals),'sha256':sha(SIGNALS)},{'input_role':'18xB_protocol','path':str(XB_PROTOCOL.relative_to(ROOT)),'rows':1,'sha256':sha(XB_PROTOCOL)},{'input_role':'18s_contract_outcome_panel_controlled_unblind','path':str(OUTCOMES.relative_to(ROOT)),'rows':len(outcomes),'sha256':sha(OUTCOMES)}]).to_csv(OUT/'18xC_source_inventory.csv',index=False)
(OUT/'18xC_environment.json').write_text(json.dumps({'generated_at_utc':datetime.now(UTC).isoformat(),'python':sys.version,'platform':platform.platform(),'pandas':pd.__version__,'numpy':np.__version__,'revision':'v2'},indent=2),encoding='utf-8')
lines=['# 18xC controlled-unblinding trading simulation','','**PASS**','','| Strategy | Block | Trades | Net PnL at 0.01 cost | Return on capital | Hit rate | Max drawdown |','|---|---|---:|---:|---:|---:|---:|']
for r in primary.sort_values(['strategy_role','evaluation_block']).itertuples(index=False): lines.append(f'| {r.strategy_role} | {r.evaluation_block} | {int(r.trade_count)} | {r.total_net_pnl:.4f} | {r.return_on_capital:.4f} | {r.hit_rate:.4f} | {r.max_drawdown:.4f} |')
lines+=['','The primary empirical strategy is profitable on the ten-date internal holdout but loses on the June external block. The GP family shows the same sign reversal, while the tree family is negative in both blocks.']
(REPORT/'18xC_controlled_unblinding_trading_simulation_report.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
write_manifest(OUT,REPORT,'18xC_sha256_manifest.csv'); print(json.dumps(summary_json,indent=2)); print('18xC PASS')

{
  "step": "18xC",
  "generated_at_utc": "2026-07-22T15:21:47.559352+00:00",
  "verdict": "PASS",
  "frozen_signal_rows": 115,
  "trade_signals": 97,
  "cost_sensitivity_trade_rows": 575,
  "strategy_cost_summary_rows": 30,
  "primary_cost_summary_rows": 6,
  "primary_cost_results": [
    {
      "strategy_role": "GAUSSIAN_PROCESS_FAMILY",
      "candidate_id": "gp_matern32_rule",
      "evaluation_block": "EXTERNAL_TEST",
      "trade_count": 22,
      "total_net_pnl": -1.063,
      "return_on_capital": -0.5152690256907416,
      "hit_rate": 0.045454545454545456,
      "max_drawdown": -1.1975
    },
    {
      "strategy_role": "GAUSSIAN_PROCESS_FAMILY",
      "candidate_id": "gp_matern32_rule",
      "evaluation_block": "INTERNAL_HOLDOUT",
      "trade_count": 5,
      "total_net_pnl": 0.5165,
      "return_on_capital": 0.3481631277384563,
      "hit_rate": 0.4,
      "max_drawdown": -0.3730000000000001
    },
    {
      "strategy_role": "PRIMARY_OVERALL",
      "candidate_id": "po